# 📈 04 - Đánh giá Model (Evaluation)
**EduTalk HUIT — Hệ thống Tư vấn Ngành học**

Notebook này thực hiện:
- Đánh giá model trên tập Test (unseen data)
- Confusion Matrix
- Precision / Recall / F1 per class
- Feature Importance (XGBoost)
- So sánh XGBoost vs Cosine Similarity
- Kết luận và hướng cải thiện

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.metrics.pairwise import cosine_similarity

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_theme(style='whitegrid')
print('✅ Libraries loaded')

## 1. Load model & data

In [ ]:
xgb_model      = joblib.load('../models/xgboost_model.pkl')
cosine_profiles= joblib.load('../models/cosine_profiles.pkl')
le             = joblib.load('../models/label_encoder.pkl')

X_test = pd.read_csv('../data/processed/X_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

print(f'Test samples: {len(X_test)}')

## 2. Đánh giá XGBoost trên tập Test

In [ ]:
y_pred = xgb_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f'🎯 Test Accuracy: {acc:.4f} ({acc*100:.2f}%)')
print('\n--- Classification Report ---')
print(classification_report(y_test, y_pred, target_names=le.classes_))

## 3. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(18, 14))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax, xticks_rotation=60, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — XGBoost (Test Set)', fontsize=14, pad=15)
plt.tight_layout()
plt.savefig('../data/samples/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature Importance

In [ ]:
importance = pd.Series(
    xgb_model.feature_importances_,
    index=X_test.columns
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, max(4, len(importance)*0.4)))
importance.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Feature Importance — XGBoost')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('../data/samples/feature_importance.png', dpi=150)
plt.show()

## 5. So sánh XGBoost vs Cosine Similarity

In [ ]:
# Cosine Similarity top-1 accuracy
sims = cosine_similarity(X_test.values, cosine_profiles.values)
y_pred_cosine = sims.argmax(axis=1)
cosine_acc = accuracy_score(y_test, y_pred_cosine)

# Cosine top-3 accuracy
top3_correct = sum(
    y_test.iloc[i] in sims[i].argsort()[::-1][:3]
    for i in range(len(y_test))
)
cosine_top3_acc = top3_correct / len(y_test)

results = {
    'Model':    ['XGBoost', 'Cosine Similarity (top-1)', 'Cosine Similarity (top-3)'],
    'Accuracy': [acc, cosine_acc, cosine_top3_acc]
}
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

# Visualization
ax = df_results.plot(kind='bar', x='Model', y='Accuracy',
                     legend=False, color=['#2563EB', '#0F766E', '#0F766E'],
                     ylim=(0, 1.0), rot=0, figsize=(8, 5))
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01,
            f'{bar.get_height():.2%}', ha='center', fontsize=11)
ax.set_title('So sánh độ chính xác: XGBoost vs Cosine Similarity')
ax.set_ylabel('Accuracy')
plt.tight_layout()
plt.savefig('../data/samples/model_comparison.png', dpi=150)
plt.show()

## ✅ Kết luận

*(Điền sau khi chạy xong)*

| Tiêu chí | XGBoost | Cosine Similarity |
|----------|---------|------------------|
| Test Accuracy | ... | ... |
| Top-3 Accuracy | - | ... |
| Thời gian train | ... | N/A |
| Ưu điểm | Chính xác cao | Không cần train |
| Nhược điểm | Cần dữ liệu nhiều | Kém hơn XGB |

**→ Hệ thống EduTalk dùng kết hợp cả 2:** XGBoost cho kết quả chính, Cosine Similarity để giải thích lý do gợi ý.

**Hướng cải thiện:**
- Thu thập thêm dữ liệu (hiện tại có ... mẫu)
- Thử SMOTE để xử lý mất cân bằng nhãn
- Thử LightGBM / Neural Network